# 3 – Erweiterter RAG-Graph: Reranking, Relevanz-Gate & Query-Reformulierung

In Notebook 2 haben wir einen linearen Graphen gebaut: `retrieve → generate → END`.  
Wer das Mermaid-Diagramm aufmerksam betrachtet hat, dem ist aufgefallen:  
**Das hätte man auch ohne LangGraph als einfache Chain lösen können.**

Dieses Notebook zeigt, warum LangGraph wirklich interessant wird:

- **Reranking** – ein Cross-Encoder bewertet die gefundenen Chunks nochmal neu
- **Relevanz-Gate** – eine bedingte Kante entscheidet: *Sind die Chunks gut genug?*
- **Query-Reformulierung** – wenn nicht, wird die Frage umformuliert und erneut gesucht

Der Graph hat jetzt **bedingte Kanten** und einen **Zyklus** – Dinge, die eine lineare Chain nicht kann.

```
retrieve → rerank → check_relevance ─(relevant)─→ generate → END
                          │
                    (nicht relevant)
                          │
                          ▼
                     reformulate ──→ retrieve  (Zyklus, max. 2×)
```

> **Voraussetzung:** Notebook 1 (Indexing) muss vorher ausgeführt worden sein.

## Imports

In [ ]:
import os
import base64
import getpass
from typing import List, TypedDict

import gradio as gr
from IPython import display

from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# Neu: Cross-Encoder für Reranking
from sentence_transformers import CrossEncoder

## Embedding-Modell & Vektordatenbank (via LM Studio, identisch zu Notebook 1)

In [ ]:
LM_STUDIO_URL  = "http://127.0.0.1:1234/v1"
EMBEDDING_MODEL = "text-embedding-multilingual-e5-large-instruct"  # Modellname wie in LM Studio geladen

DB_DIR          = "./chroma_db"


class E5Embeddings(OpenAIEmbeddings):
    """Wrapper, der die vom E5-Modell erwarteten Präfixe automatisch setzt."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


embeddings = E5Embeddings(
    model=EMBEDDING_MODEL,
    api_key="lm-studio",   # LM Studio braucht keinen echten Key
    base_url=LM_STUDIO_URL,
    check_embedding_ctx_length=False,
)

In [ ]:
vectorstore = Chroma(
    persist_directory=DB_DIR,
    embedding_function=embeddings,
)

print(f"✅ Vektordatenbank geladen – {vectorstore._collection.count()} Chunks verfügbar.")

## Reranking-Modell laden

Ein **Bi-Encoder** (unser E5-Modell) ist schnell, weil Query und Dokument getrennt eingebettet werden.  
Ein **Cross-Encoder** ist langsamer, aber genauer: er bewertet Query und Dokument *gemeinsam*.

Strategie: Erst viele Kandidaten mit dem Bi-Encoder holen, dann mit dem Cross-Encoder die besten auswählen.

> **Wichtig:** Da unsere Dokumente auf Deutsch sind, brauchen wir einen **multilingualen** Cross-Encoder.  
> Ein rein englischer Reranker (z.B. `ms-marco-MiniLM`) liefert bei deutschen Texten systematisch zu niedrige Scores.

In [ ]:
RERANKER_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker = CrossEncoder(RERANKER_MODEL, max_length=512)

print(f"✅ Reranker geladen: {RERANKER_MODEL}")

## LLM konfigurieren

In [ ]:
#DEEPINFRA_API_KEY = getpass.getpass("DeepInfra API-Key eingeben: ")
#
#llm = ChatOpenAI(
#    model_name="meta-llama/Llama-3.3-70B-Instruct-Turbo",
#    openai_api_key=DEEPINFRA_API_KEY,
#    openai_api_base="https://api.deepinfra.com/v1/openai",
#    max_tokens=5000,
#    temperature=0,
#)

In [ ]:
LM_STUDIO_LLM_URL = "http://localhost:1234/v1"  # Gleiche URL, anderer Modell-Slot

llm = ChatOpenAI(
    model="qwen/qwen3.5-9b",  # z.B. "llama-3.3-70b-instruct"
    api_key="lm-studio",
    base_url=LM_STUDIO_LLM_URL,
    max_tokens=5000,
    temperature=0,
)


## Konfiguration des erweiterten Graphen

In [ ]:
# --- PARAMETER ---

RETRIEVE_K        = 15      # Anzahl Kandidaten aus der Vektorsuche (breit suchen)
RERANK_TOP_N      = 5       # Davon die N besten nach Reranking behalten
RELEVANCE_THRESHOLD = 0.3   # Mindest-Score des besten Chunks (Cross-Encoder)
MAX_RETRIES       = 2       # Maximale Query-Reformulierungen, bevor wir aufgeben

## State – jetzt mit zusätzlichen Feldern

Gegenüber Notebook 2 kommen hinzu:
- `rerank_scores` – die Bewertungen des Cross-Encoders
- `retry_count` – zählt, wie oft die Query reformuliert wurde (Endlosschleifen verhindern!)

In [ ]:
class GraphState(TypedDict):
    question:      str
    context:       List[str]
    metadata:      List[dict]
    rerank_scores: List[float]
    answer:        str
    token_usage:   dict
    retry_count:   int

## Nodes – die Bausteine des Graphen

Vier Knoten statt zwei:

| Node | Aufgabe |
|------|------|
| `retrieve` | Holt Kandidaten-Chunks aus der Vektorsuche |
| `rerank` | Bewertet die Kandidaten mit dem Cross-Encoder und behält die besten |
| `generate` | Erzeugt die Antwort mit dem LLM |
| `reformulate` | Formuliert die Frage um, wenn der Kontext nicht relevant genug war |

In [ ]:
# --- NODE: RETRIEVE ---

def retrieve(state: GraphState) -> dict:
    """Holt Kandidaten-Chunks aus der Vektordatenbank (breite Suche)."""
    print(f"--- RETRIEVE (Versuch {state.get('retry_count', 0) + 1}) ---")
    print(f"    Query: {state['question']}")

    docs = vectorstore.similarity_search(state["question"], k=RETRIEVE_K)

    context  = []
    metadata = []

    for i, doc in enumerate(docs):
        context.append(doc.page_content)
        source_file = os.path.basename(doc.metadata.get("source", "Unbekannt"))
        page_num    = doc.metadata.get("page", 0) + 1
        metadata.append({"id": i + 1, "source": source_file, "page": page_num})

    print(f"    → {len(docs)} Kandidaten gefunden")
    return {"context": context, "metadata": metadata}

In [ ]:
# --- NODE: RERANK ---

def rerank(state: GraphState) -> dict:
    """Bewertet die Kandidaten mit einem Cross-Encoder und behält die Top-N."""
    print("--- RERANK ---")
    question = state["question"]

    # Cross-Encoder erwartet Paare aus (Query, Passage)
    pairs = [(question, chunk) for chunk in state["context"]]
    scores = reranker.predict(pairs)

    # Nach Score sortieren (absteigend) und Top-N behalten
    scored_items = sorted(
        zip(scores, state["context"], state["metadata"]),
        key=lambda x: x[0],
        reverse=True,
    )

    top_items = scored_items[:RERANK_TOP_N]

    # IDs neu vergeben (1-basiert)
    reranked_context  = []
    reranked_metadata = []
    reranked_scores   = []

    for new_id, (score, text, meta) in enumerate(top_items, start=1):
        reranked_context.append(text)
        reranked_metadata.append({**meta, "id": new_id})
        reranked_scores.append(float(score))

    print(f"    → Top-{RERANK_TOP_N} Scores: {[f'{s:.3f}' for s in reranked_scores]}")

    return {
        "context":       reranked_context,
        "metadata":      reranked_metadata,
        "rerank_scores": reranked_scores,
    }

In [ ]:
# --- NODE: GENERATE ---

ANSWER_PROMPT = ChatPromptTemplate.from_template("""\
Du bist ein präziser Assistent. Beantworte die Frage NUR basierend auf dem KONTEXT.

REGELN:
1. Verweise im Text deiner Antwort auf die Abschnitte, z.B. [1] oder [Quelle: Datei.pdf, S. 5].
2. Wenn die Info nicht im Kontext ist, sag es offen.
3. Erfinde KEINE Fakten.

KONTEXT:
{context}

FRAGE: {question}
""")


def generate(state: GraphState) -> dict:
    """Erzeugt eine Antwort auf Basis der besten Kontext-Chunks."""
    print("--- GENERATE ---")

    formatted_context = ""
    for i, text in enumerate(state["context"]):
        meta  = state["metadata"][i]
        score = state["rerank_scores"][i]
        formatted_context += (
            f"\n--- ABSCHNITT {meta['id']} "
            f"(Quelle: {meta['source']}, Seite {meta['page']}, "
            f"Relevanz: {score:.3f}) ---\n"
            f"{text}\n"
        )

    chain    = ANSWER_PROMPT | llm
    response = chain.invoke({"context": formatted_context, "question": state["question"]})
    usage    = response.response_metadata.get("token_usage", {})

    return {"answer": response.content, "token_usage": usage}

In [ ]:
# --- NODE: REFORMULATE ---

REFORMULATE_PROMPT = ChatPromptTemplate.from_template("""\
Die folgende Suchanfrage hat keine ausreichend relevanten Ergebnisse geliefert.
Formuliere die Anfrage um – nutze Synonyme, andere Formulierungen oder zerlege sie
in einen präziseren Kern. Antworte NUR mit der neuen Suchanfrage, ohne Erklärung.

Ursprüngliche Anfrage: {question}
""")


def reformulate(state: GraphState) -> dict:
    """Formuliert die Frage um, damit die nächste Suche bessere Ergebnisse liefert."""
    print("--- REFORMULATE ---")

    chain        = REFORMULATE_PROMPT | llm
    response     = chain.invoke({"question": state["question"]})
    new_question = response.content.strip()

    retry_count = state.get("retry_count", 0) + 1
    print(f"    → Neue Query: '{new_question}' (Versuch {retry_count})")

    return {"question": new_question, "retry_count": retry_count}

## Bedingte Kante – das Herzstück

Hier passiert das, was eine einfache Chain nicht kann:  
**Der Graph entscheidet zur Laufzeit, welchen Weg er nimmt.**

Die Funktion `check_relevance` prüft den besten Reranking-Score:  
- Über dem Schwellwert → weiter zu `generate`  
- Darunter (und noch Versuche übrig) → zurück zu `reformulate` → `retrieve` (Zyklus!)

In [ ]:
# --- CONDITIONAL EDGE ---

def check_relevance(state: GraphState) -> str:
    """Entscheidet, ob der Kontext relevant genug ist oder reformuliert werden muss."""
    best_score  = max(state["rerank_scores"]) if state["rerank_scores"] else 0.0
    retry_count = state.get("retry_count", 0)

    print(f"--- CHECK RELEVANCE ---")
    print(f"    Bester Score: {best_score:.3f} (Schwelle: {RELEVANCE_THRESHOLD})")
    print(f"    Bisherige Versuche: {retry_count} / {MAX_RETRIES}")

    if best_score >= RELEVANCE_THRESHOLD:
        print("    → RELEVANT – weiter zu Generate")
        return "relevant"
    elif retry_count < MAX_RETRIES:
        print("    → NICHT RELEVANT – Query wird reformuliert")
        return "not_relevant"
    else:
        print("    → NICHT RELEVANT, aber max. Versuche erreicht – Generate mit bestem Ergebnis")
        return "relevant"  # Fallback: trotzdem antworten

## Graph zusammenbauen

Jetzt verbinden wir alles. Beachte den Unterschied zu Notebook 2:  
- `add_conditional_edges` statt `add_edge` nach dem Reranking  
- Ein Zyklus: `reformulate → retrieve → rerank → check_relevance → ...`

In [ ]:
# --- GRAPH ZUSAMMENBAUEN ---

workflow = StateGraph(GraphState)

# Knoten registrieren
workflow.add_node("retrieve_node",    retrieve)
workflow.add_node("rerank_node",      rerank)
workflow.add_node("generate_node",    generate)
workflow.add_node("reformulate_node", reformulate)

# Kanten definieren
workflow.set_entry_point("retrieve_node")
workflow.add_edge("retrieve_node", "rerank_node")

# ⭐ Die bedingte Kante: check_relevance entscheidet den Weg
workflow.add_conditional_edges(
    "rerank_node",          # Nach diesem Knoten ...
    check_relevance,         # ... wird diese Funktion aufgerufen ...
    {                        # ... und ihr Rückgabewert bestimmt den nächsten Knoten:
        "relevant":     "generate_node",
        "not_relevant": "reformulate_node",
    },
)

# Der Zyklus: reformulate → retrieve (und von dort wieder rerank → check)
workflow.add_edge("reformulate_node", "retrieve_node")
workflow.add_edge("generate_node", END)

# Graph kompilieren
app = workflow.compile()

## Graph visualisieren

Vergleiche dieses Diagramm mit dem aus Notebook 2.  
Hier sieht man den Unterschied: **bedingte Verzweigung und ein Zyklus**.

In [ ]:
def display_graph(graph_app):
    """Zeigt den LangGraph als Mermaid-Diagramm an und speichert die Syntax."""
    mermaid_code = graph_app.get_graph().draw_mermaid()

    encoded = base64.b64encode(mermaid_code.encode()).decode()
    display.display(display.Image(url=f"https://mermaid.ink/img/{encoded}"))

    with open("rag_graph_erweitert.mmd", "w", encoding="utf-8") as f:
        f.write(mermaid_code)


display_graph(app)

## Gradio-Interface

In [ ]:
def chat_interface(question: str) -> str:
    """Verarbeitet eine Frage über den erweiterten RAG-Graphen."""
    result = app.invoke({"question": question, "retry_count": 0})

    # Antwort
    answer = result["answer"]

    # Reranking-Info
    scores = result.get("rerank_scores", [])
    rerank_info = "\n\n---\n🎯 **Reranking-Scores (Top-Chunks):**\n"
    for i, (meta, score) in enumerate(zip(result["metadata"], scores)):
        rerank_info += f"- [{meta['id']}] {meta['source']} (S. {meta['page']}): {score:.3f}\n"

    # Token-Statistik
    usage = result.get("token_usage", {})
    token_info = (
        f"\n📊 **Token-Statistik:**\n"
        f"- Input: {usage.get('prompt_tokens', 'N/A')}\n"
        f"- Output: {usage.get('completion_tokens', 'N/A')}\n"
        f"- Gesamt: {usage.get('total_tokens', 'N/A')}"
    )

    # Retry-Info
    retries = result.get("retry_count", 0)
    retry_info = ""
    if retries > 0:
        retry_info = f"\n\n🔄 Query wurde {retries}× reformuliert."

    return answer + rerank_info + token_info + retry_info


demo = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs="text",
    title="Erweiterter RAG mit Reranking & Relevanz-Gate",
    description="Fragen an deine PDF-Dokumente – jetzt mit Cross-Encoder-Reranking und automatischer Query-Reformulierung.",
    flagging_mode="never",
)

demo.launch()

## Zum Nachdenken

- Wie verändert das Reranking die Qualität der Antworten im Vergleich zu Notebook 2?
- Was passiert, wenn du den `RELEVANCE_THRESHOLD` auf 0.8 setzt? Auf 0.01?
- Wann wird die Query-Reformulierung tatsächlich ausgelöst?
- Welche weiteren Knoten oder Bedingungen könnten sinnvoll sein?